# Lakekeeper + Iceberg quick tour
Run Spark against the Lakekeeper REST catalog, write a small Iceberg table to MinIO, and verify what was written.

What you will do:
- configure Spark for the Lakekeeper REST catalog and MinIO S3 endpoint
- create a tiny visits dataset
- write/read the Iceberg table and check where the files land


## 1) Start Spark with Lakekeeper + MinIO
All parameters live at the top so you can tweak the catalog, namespace, table name, or credentials if you changed defaults.


In [1]:
import os
from pyspark.sql import SparkSession, functions as F

catalog = "lk"
namespace = "demo"
table = "visits"
full_table = f"{catalog}.{namespace}.{table}"

access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://127.0.0.1:9000")
hconf.set("fs.s3a.access.key", access_key)
hconf.set("fs.s3a.secret.key", secret_key)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.connection.ssl.enabled", "false")

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {catalog}.{namespace}")

print("Catalog REST endpoint: http://127.0.0.1:8181/catalog")
print(f"Warehouse bucket: s3://{warehouse}/warehouse")
print(f"Target table: {full_table}")


Catalog REST endpoint: http://127.0.0.1:8181/catalog
Warehouse bucket: s3://mydatalab/warehouse
Target table: lk.demo.visits


## 2) Build a sample dataset
A few synthetic visit events to make the table easy to inspect.


In [2]:
data = [
    ("2025-01-01", "alice", "web", 3),
    ("2025-01-01", "bob", "email", 1),
    ("2025-01-02", "alice", "ads", 2),
    ("2025-01-02", "carol", "web", 5),
]

df = (
    spark.createDataFrame(data, ["event_date", "user_id", "source", "visits"])
    .withColumn("ingested_at", F.current_timestamp())
)

print("Preview before writing:")
df.orderBy("event_date", "user_id").show(truncate=False)


Preview before writing:
+----------+-------+------+------+--------------------------+
|event_date|user_id|source|visits|ingested_at               |
+----------+-------+------+------+--------------------------+
|2025-01-01|alice  |web   |3     |2025-11-25 01:29:57.569736|
|2025-01-01|bob    |email |1     |2025-11-25 01:29:57.569736|
|2025-01-02|alice  |ads   |2     |2025-11-25 01:29:57.569736|
|2025-01-02|carol  |web   |5     |2025-11-25 01:29:57.569736|
+----------+-------+------+------+--------------------------+



## 3) Write as Iceberg and query it
We use `createOrReplace` so rerunning the cell keeps the notebook idempotent.


In [3]:
(
    df.writeTo(full_table)
    .using("iceberg")
    .createOrReplace()
)

print("Current table contents:")
spark.table(full_table).orderBy("event_date", "user_id").show(truncate=False)

print("Visits by date:")
spark.sql(
    f'''
    SELECT event_date, SUM(visits) AS total_visits
    FROM {full_table}
    GROUP BY event_date
    ORDER BY event_date
    '''
).show()


Current table contents:
+----------+-------+------+------+--------------------------+
|event_date|user_id|source|visits|ingested_at               |
+----------+-------+------+------+--------------------------+
|2025-01-01|alice  |web   |3     |2025-11-25 01:30:02.021395|
|2025-01-01|bob    |email |1     |2025-11-25 01:30:02.021395|
|2025-01-02|alice  |ads   |2     |2025-11-25 01:30:02.021395|
|2025-01-02|carol  |web   |5     |2025-11-25 01:30:02.021395|
+----------+-------+------+------+--------------------------+

Visits by date:
+----------+------------+
|event_date|total_visits|
+----------+------------+
|2025-01-01|           4|
|2025-01-02|           7|
+----------+------------+



## 4) Inspect where the data landed
The `Location` entry in Iceberg metadata points to the files MinIO is storing.


In [4]:
location_row = (
    spark.sql(f"DESCRIBE TABLE EXTENDED {full_table}")
    .filter("col_name = 'Location'")
    .collect()
)
if location_row:
    print(f"Table data lives in MinIO at: {location_row[0].data_type}")
else:
    print("Location not found in table metadata.")

spark.read.format("iceberg").load(full_table).orderBy("ingested_at").show(truncate=False)


Table data lives in MinIO at: s3://mydatalab/warehouse/019ab8a1-b5c9-72a3-8d81-dfbd1bd284ab/019ab8a1-c840-7c42-b72e-51535e3f05b9
+----------+-------+------+------+--------------------------+
|event_date|user_id|source|visits|ingested_at               |
+----------+-------+------+------+--------------------------+
|2025-01-01|alice  |web   |3     |2025-11-25 01:30:02.021395|
|2025-01-01|bob    |email |1     |2025-11-25 01:30:02.021395|
|2025-01-02|alice  |ads   |2     |2025-11-25 01:30:02.021395|
|2025-01-02|carol  |web   |5     |2025-11-25 01:30:02.021395|
+----------+-------+------+------+--------------------------+



## Next steps
- adjust the dataset or add updates to watch how Iceberg snapshots change
- switch the namespace/table names to keep multiple sandboxes side by side
- explore `DESCRIBE HISTORY lk.demo.visits` or `lk.system.snapshots` for time-travel details
